# Direct Detection

In [1]:
data_LZ_2024 = np.loadtxt("data/LZ2024_Limit.csv", delimiter=',')

mass_data_LZ_2024 = data_LZ_2024[:, 0]
sigmasidatacm2_LZ_2024 = data_LZ_2024[:, 1]

cm2_to_GeVm2 = 2.5681894616e27          # 1 cm^2 = (hbar c)^-2 GeV^-2
sigmasidataGeV_LZ_2024 = sigmasidatacm2_LZ_2024 * cm2_to_GeVm2

# Interpolating
dd_limit_fun_LZ2024 = interpolate.interp1d(mass_data_LZ_2024, sigmasidataGeV_LZ_2024)

NameError: name 'np' is not defined

In [ ]:
mp = 0.93827208816  # proton mass in GeV

def sig_si_theo(ma, alpha, alphaD, mchi, epsilon):
    """SI DM-nucleon cross section, in GeV^-2."""
    mu = mchi*mp/(mchi + mp)                                   # reduced mass
    return 16*np.pi * epsilon**2 * alpha * alphaD * mu**2 / ma**4

def eps_dd_limit_LZ2024(ma, alpha, alphaD, mchi):
    """epsilon at which sig_si_theo == dd_limit_fun_LZ2024(mchi).

    Points whose mchi falls outside the tabulated LZ mass range are dropped,
    so the limit is only drawn where the experiment actually constrains it.

    Returns (ma_valid, eps_valid): the truncated mediator mass range and the
    corresponding epsilon limits.
    """
    ma, mchi = np.broadcast_arrays(np.atleast_1d(ma), np.atleast_1d(mchi))

    # Keep only the points the interpolator can evaluate:
    inside = (mchi >= mass_data_LZ_2024.min()) & (mchi <= mass_data_LZ_2024.max())
    ma, mchi = ma[inside], mchi[inside]

    sigma_lim = dd_limit_fun_LZ2024(mchi)                      # GeV^-2
    return ma, np.sqrt(sigma_lim / sig_si_theo(ma, alpha, alphaD, mchi, 1.0))


In [ ]:
mlims, eplims =eps_dd_limit_LZ2024(ma, alpha, alphaD, mchi)

In [ ]:
## Creating a plot:

fig, ax = plt.subplots(figsize=(10, 10))

ax.set_xlabel(r"$m_{A'}$(GeV)")
ax.set_ylabel(r"$\epsilon$")

## Log Axis
ax.set_xscale('log')
ax.set_yscale('log')

## Lock the square shape
#ax.set_aspect('equal', adjustable='datalim')

## Major grid:
ax.grid(True, which='major', linestyle='-', linewidth=0.75, alpha=0.75)

## Minor ticks and grid:
ax.minorticks_on()
ax.grid(True, which='minor', linestyle='-', linewidth=0.5, alpha=0.75)

ax.set_axisbelow(True) # Ensure grid is below data

ax.plot(ma, vare_1, color = 'black', linewidth = 2.5, label = r"Correct DM abundance")

ax.plot(mlims, eplims, color = 'red', linewidth = 2.5, label = r"LZ2024")

ax.fill_between(mlims, eplims, 
                1,
                label= rf"Direct detection (Xenon1T)",
                color='red', 
                alpha=0.15, 
                zorder=1)

## Color palets:

dataset_colors = ['#9671bd', '#7e7e7e', '#77b5b6']
dataset_line_colors = ['#6a408d', '#4e4e4e', '#378d94']

# And the colour for the lines (regressions):
regression_color = '#8a8a8a' # <-- neutral grey
# regression_color = '#7f9fa1' # <-- greyish seafoam

'''

ax.scatter(final_data[:,0],final_data[:,1],
            label = "Correct DM abundance",
          s = 90,
          color = dataset_colors[0],
          edgecolors = dataset_line_colors[0],
          linewidths = 1.5,
          zorder = 3
    )

'''

## Setting a costumised legend:
handles, labels = ax.get_legend_handles_labels() # get all legend items
desired_order = [0,1]  # change the order of legend elements


ax.legend(
    [handles[i] for i in desired_order],
    [labels[i] for i in desired_order],
    loc = 'upper center',
    bbox_to_anchor = (0.5, 1.10),  # center top, above axes
    ncol = 4,                      # spread horizontally
    frameon = False                # removes legend border
)

# Define how much to zoom out from the data plotting range (cm):
x_min = 1e-2
x_max = 2e1
x_range = x_max - x_min

y_min = 1e-7
y_max = 1e0
y_range = y_max - y_min

# Set the new plotting limits explicitly
ax.set_xbound(x_min, x_max)
ax.set_ybound(y_min, y_max)


# Save and show the figure:
plt.savefig("figures/relic_abundance_0.pdf", bbox_inches='tight')
plt.show()

In [ ]:
### XENONnT SR0+1 SI limit (FIG 3 A of arXiv:2409.17868), from the .npy points

limits_dir_XENONnT = ("data/XENONnT-light_wimp_data_release-64d2e13/"
                      "light_wimp_data_release/limits")

# The published limit is the power-constrained one; the folder also holds
# xenonnt_si_limit_before_pcl.npy and xenonnt_si_limit_after_dl_pcl.npy.
data_XENONnT_si = np.load(f"{limits_dir_XENONnT}/xenonnt_si_limit_after_pcl.npy")

mass_data_XENONnT_si = data_XENONnT_si['mass_gev']
sigmasidatacm2_XENONnT_si = data_XENONnT_si['limit_cm2']

sigmasidataGeV_XENONnT_si = sigmasidatacm2_XENONnT_si * cm2_to_GeVm2

# Interpolating: only 7 tabulated points spanning 5 decades in sigma, so a
# linear interpolation would overshoot by up to ~5x between the low-mass
# points. The limit is close to a straight line in log-log, so interpolate there.
log_dd_limit_XENONnT_si = interpolate.interp1d(np.log(mass_data_XENONnT_si),
                                               np.log(sigmasidataGeV_XENONnT_si))

def dd_limit_fun_XENONnT_si(mchi):
    return np.exp(log_dd_limit_XENONnT_si(np.log(mchi)))

In [ ]:
def eps_dd_limit_XENONnT_si(ma, alpha, alphaD, mchi):
    """epsilon at which sig_si_theo == dd_limit_fun_XENONnT_si(mchi).

    Same construction as eps_dd_limit_LZ2024, against the XENONnT SR0+1
    spin-independent limit. Points whose mchi falls outside the tabulated
    3-12 GeV range are dropped.

    Returns (ma_valid, eps_valid).
    """
    ma, mchi = np.broadcast_arrays(np.atleast_1d(ma), np.atleast_1d(mchi))

    # Keep only the points the interpolator can evaluate:
    inside = ((mchi >= mass_data_XENONnT_si.min())
              & (mchi <= mass_data_XENONnT_si.max()))
    ma, mchi = ma[inside], mchi[inside]

    sigma_lim = dd_limit_fun_XENONnT_si(mchi)                  # GeV^-2
    return ma, np.sqrt(sigma_lim / sig_si_theo(ma, alpha, alphaD, mchi, 1.0))

In [ ]:
mlims_XENONnT, eplims_XENONnT = eps_dd_limit_XENONnT_si(ma, alpha, alphaD, mchi)

In [ ]:
## Creating a plot:

fig, ax = plt.subplots(figsize=(10, 10))

ax.set_xlabel(r"$m_{A'}$(GeV)")
ax.set_ylabel(r"$\epsilon$")

## Log Axis
ax.set_xscale('log')
ax.set_yscale('log')

## Major grid:
ax.grid(True, which='major', linestyle='-', linewidth=0.75, alpha=0.75)

## Minor ticks and grid:
ax.minorticks_on()
ax.grid(True, which='minor', linestyle='-', linewidth=0.5, alpha=0.75)

ax.set_axisbelow(True) # Ensure grid is below data

ax.plot(ma, vare_1, color = 'black', linewidth = 2.5, label = r"Correct DM abundance")

ax.plot(mlims, eplims, color = 'red', linewidth = 2.5, label = r"LZ 2024")

ax.fill_between(mlims, eplims,
                1,
                color='red',
                alpha=0.15,
                zorder=1)

ax.plot(mlims_XENONnT, eplims_XENONnT,
        color = '#2a78d6', linewidth = 2.5, label = r"XENONnT 2025")

ax.fill_between(mlims_XENONnT, eplims_XENONnT,
                1,
                color='#2a78d6',
                alpha=0.15,
                zorder=1)

## Setting a costumised legend:
handles, labels = ax.get_legend_handles_labels() # get all legend items
desired_order = [0,1,2]  # change the order of legend elements

ax.legend(
    [handles[i] for i in desired_order],
    [labels[i] for i in desired_order],
    loc = 'upper center',
    bbox_to_anchor = (0.5, 1.10),  # center top, above axes
    ncol = 4,                      # spread horizontally
    frameon = False                # removes legend border
)

# Define how much to zoom out from the data plotting range (cm):
x_min = 1e-2
x_max = 2e1

y_min = 1e-7
y_max = 1e-1

# Set the new plotting limits explicitly
ax.set_xbound(x_min, x_max)
ax.set_ybound(y_min, y_max)

# Save and show the figure:
plt.savefig("figures/relic_abundance_XENONnT_LZ.pdf", bbox_inches='tight')
plt.show()